# Manufacturing Stress Adaptive-Agent Workbench

A companion to `manufacturing_stress_hybrid_workbench` for inspecting the persistent adaptive strategy, reviewing durable mutations, and generating executive-ready adaptation reports. This notebook intentionally does not run forecasts, backtests, LLM calls, or strategy mutations.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "uv.lock").exists() and (p / "implementations").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Repository root not found.")
for package_root in (REPO_ROOT / "aieng-forecasting", REPO_ROOT / "implementations"):
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))
load_dotenv(REPO_ROOT / ".env", override=False)

RUN_ADAPTIVE_AGENT = False
RUN_ADAPTIVE_BACKTEST = False
LIVE_ADAPTATION = False
REFRESH_INPUT_DATA = False
AS_OF = "2017-01-01"
BACKTEST_STRIDE = 3
FORCE_REFRESH_AGENT_CACHE = False
STRATEGY_DIR = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "adaptive_agent" / "skills" / "manufacturing-strategy"
REPORT_DIR = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "reports" / "adaptive_agent_workbench"
STORE_DIR = REPO_ROOT / "data" / "predictions"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print({"run_adaptive_agent": RUN_ADAPTIVE_AGENT, "run_adaptive_backtest": RUN_ADAPTIVE_BACKTEST, "live_adaptation": LIVE_ADAPTATION, "as_of": AS_OF})

## Load data, task specification, and strategy state

The notebook uses the same cutoff-aware manufacturing service and smoke task specification as the hybrid workbench. Adaptive execution is opt-in.

In [ ]:
from aieng.forecasting.evaluation import BacktestSpec
from manufacturing_stress_forecasting.data import build_manufacturing_stress_service
from manufacturing_stress_forecasting.adaptive_agent.state import ManufacturingStrategyState

SPEC_PATH = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
with SPEC_PATH.open(encoding="utf-8") as file:
    base_spec = BacktestSpec.model_validate(yaml.safe_load(file))
spec = base_spec.model_copy(update={"stride": BACKTEST_STRIDE})
service = build_manufacturing_stress_service(cache_dir=REPO_ROOT / "data" / "fred", yahoo_cache_dir=REPO_ROOT / "data" / "yfinance", refresh=REFRESH_INPUT_DATA)
context = service.context(as_of=pd.Timestamp(AS_OF).to_pydatetime())
state_path = STRATEGY_DIR / "skill_state.yaml"
state = ManufacturingStrategyState.model_validate(yaml.safe_load(state_path.read_text(encoding="utf-8")))
state_json = json.dumps(state.model_dump(mode="json"), sort_keys=True, separators=(",", ":"))
state_hash = hashlib.sha256(state_json.encode()).hexdigest()
print({"state_hash": state_hash, "strategy_dir": str(STRATEGY_DIR), "target": spec.task.target_series_id})
display(pd.DataFrame([{"status": h.status, "statement": h.statement} for h in state.hypotheses]))

## Optional adaptive-agent execution

Set `RUN_ADAPTIVE_AGENT = True` to execute the stateful agent. The agent is built with its durable strategy mutation tools; leave the toggle false for inspection-only runs.

In [ ]:
if RUN_ADAPTIVE_AGENT:
    from manufacturing_stress_forecasting.adaptive_agent.agent import build_manufacturing_adaptive_agent_predictor
    adaptive_predictor = build_manufacturing_adaptive_agent_predictor(strategy_dir=STRATEGY_DIR)
    adaptive_result = adaptive_predictor.predict(spec.task, context)[0]
    print(adaptive_result)
else:
    print("Adaptive-agent execution skipped; inspection-only mode is active.")

## Optional adaptive backtest

This uses the existing cached backtest runner and evaluates the adaptive agent across comparable origins. It remains disabled by default because it can issue multiple LLM calls and persist mutation events.

In [ ]:
if RUN_ADAPTIVE_BACKTEST:
    from manufacturing_stress_forecasting.adaptive_agent.agent import build_manufacturing_adaptive_agent_predictor
    from manufacturing_stress_forecasting.hybrid_agent import HybridAgentPredictor
    from manufacturing_stress_forecasting.run_agent_backtest import run_or_load_backtest
    adaptive_backtest_predictor = HybridAgentPredictor(
        build_manufacturing_adaptive_agent_predictor(strategy_dir=STRATEGY_DIR)
    )
    adaptive_backtest_result = run_or_load_backtest(
        adaptive_backtest_predictor,
        spec,
        service,
        force_refresh=FORCE_REFRESH_AGENT_CACHE or REFRESH_INPUT_DATA,
        store_dir=STORE_DIR,
    )
    print({"mean_score": adaptive_backtest_result.mean_score, "metric": adaptive_backtest_result.metric, "scored_origins": len(adaptive_backtest_result.scores)})
else:
    print("Adaptive backtest skipped; inspection-only mode is active.")

## Adaptation audit and executive report

In [ ]:
from manufacturing_stress_forecasting.adaptive_agent.report import load_audit, write_report

AUDIT_PATH = STRATEGY_DIR / ".history" / "adaptation_audit.jsonl"
audit = load_audit(AUDIT_PATH) if AUDIT_PATH.exists() else pd.DataFrame()
if audit.empty:
    print(f"No durable adaptation mutations recorded at {AUDIT_PATH}.")
else:
    display(audit)
    paths = write_report(AUDIT_PATH, REPORT_DIR)
    print([path.name for path in paths])

## Interpretation checklist

- Compare inspection and execution runs using the strategy hash, trace metadata, and mutation audit.
- Treat adaptation as durable only when the redacted JSONL audit and post-state hash are present.
- Review mutation activity alongside forecast quality; mutation count alone is not evidence of improvement.
- Keep `RUN_ADAPTIVE_AGENT = False` for reproducible inspection-only runs.